# Lotto Data Collection and Preprocessing

This notebook collects and preprocesses lotto history through the shared `src.pipelines` workflow.

The goal is to keep the notebook as the orchestration layer while reusing the same Python modules that the rest of the project uses.

- use `run_sync_pipeline()` when you want to refresh the canonical raw workbook
- use `run_data_pipeline()` when you want to rebuild the canonical processed CSV
- keep the lower-level preprocessing preview in the notebook so we can still inspect intermediate tables


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
APP_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src" / "notebook_support.py").exists() and (candidate / "data").exists():
        APP_ROOT = candidate
        break
    if (candidate / "app" / "src" / "notebook_support.py").exists() and (candidate / "app" / "data").exists():
        APP_ROOT = candidate / "app"
        break

if APP_ROOT is None:
    raise RuntimeError(f"Could not locate app root from {cwd}")

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

from src.notebook_support import describe_notebook_environment
describe_notebook_environment(APP_ROOT)

import pandas as pd
from src.data.load_data import load_lotto_source
from src.data.preprocess import preprocess_lotto_data
from src.data.validate_data import (
    validate_number_ranges,
    validate_processed_columns,
    validate_unique_main_numbers,
)
from src.pipelines import run_data_pipeline, run_sync_pipeline
from src.config import RAW_LOTTO_FILE, PROCESSED_LOTTO_FILE


### 2. Select the Data Source and Load Raw Data

This notebook supports multiple input modes.

- `excel`: stable default mode that loads the local Excel file
- `auto`: requests-based automatic collection path
- `auto_browser`: browser-based automatic collection path

If you want to refresh the canonical workbook first, enable `SYNC_BEFORE_LOAD`.


In [2]:
# ============================================================
# Data source selection
# ============================================================
DATA_SOURCE_MODE = "excel"
CUSTOM_RAW_FILE = RAW_LOTTO_FILE
START_ROUND = 1
END_ROUND = None
SYNC_BEFORE_LOAD = False

if SYNC_BEFORE_LOAD and DATA_SOURCE_MODE == "excel" and Path(CUSTOM_RAW_FILE) == Path(RAW_LOTTO_FILE):
    run_sync_pipeline()

df_raw = load_lotto_source(
    source=DATA_SOURCE_MODE,
    file_path=CUSTOM_RAW_FILE,
    start_round=START_ROUND,
    end_round=END_ROUND,
)

df_raw.head()


Last stored round: 1217
Checking for new rounds starting from 1218...
Fetching round 1218...
Fetching round 1219...
Fetching round 1220...
Fetching round 1221...
Fetching round 1222...
Fetching round 1223...
Round 1223 is not available. Stopping incremental sync.
Appending 5 new rows after No 1217 into /workspace/data/raw/lotto_history_latest.xlsx...


,No,회차,당첨번호1,당첨번호2,당첨번호3,당첨번호4,당첨번호5,당첨번호6,보너스,순위,당첨게임수,1게임당 당첨금액
0,1,1214,10,15,19,27,30,33,14,1등,12 명,"2,431,577,188 원"
1,2,1213,5,11,25,27,36,38,2,1등,18 명,"1,740,011,646 원"
2,3,1212,5,8,25,31,41,44,45,1등,12 명,"2,654,089,032 원"
3,4,1211,23,26,27,35,38,40,10,1등,14 명,"2,370,956,036 원"
4,5,1210,1,7,9,17,27,38,31,1등,24 명,"1,102,298,407 원"


In [3]:
df_raw.columns.tolist(), df_raw.shape

(['No',
  '회차',
  '당첨번호1',
  '당첨번호2',
  '당첨번호3',
  '당첨번호4',
  '당첨번호5',
  '당첨번호6',
  '보너스',
  '순위',
  '당첨게임수',
  '1게임당 당첨금액'],
 (1222, 12))

In [4]:
df_clean = preprocess_lotto_data(df_raw)
validate_processed_columns(df_clean)
validate_number_ranges(df_clean)
validate_unique_main_numbers(df_clean)
df_clean.head()

,row_id,round,n1,n2,n3,n4,n5,n6,bonus,rank_text,winner_count,prize_amount,numbers,sum_main,odd_count,even_count,low_count,high_count
0,1214,1,10,23,29,33,37,40,16,1등,0,0,"10,23,29,33,37,40",172,4,2,1,5
1,1213,2,9,13,21,25,32,42,2,1등,1,2002006800,"9,13,21,25,32,42",142,4,2,3,3
2,1212,3,11,16,19,21,27,31,30,1등,1,2000000000,"11,16,19,21,27,31",125,5,1,4,2
3,1211,4,14,27,30,31,40,42,2,1등,0,0,"14,27,30,31,40,42",184,2,4,1,5
4,1210,5,16,24,29,40,41,42,3,1등,0,0,"16,24,29,40,41,42",192,2,4,1,5


In [5]:
processed_output = run_data_pipeline(
    source=DATA_SOURCE_MODE,
    file_path=CUSTOM_RAW_FILE,
    start_round=START_ROUND,
    end_round=END_ROUND,
)
print(processed_output)


Last stored round: 1222
Checking for new rounds starting from 1223...
Fetching round 1223...
Round 1223 is not available. Stopping incremental sync.
No workbook updates were needed.
/workspace/data/processed/lotto_cleaned.csv


In [6]:
df_saved = pd.read_csv(PROCESSED_LOTTO_FILE)
df_saved.head()

,row_id,round,n1,n2,n3,n4,n5,n6,bonus,rank_text,winner_count,prize_amount,numbers,sum_main,odd_count,even_count,low_count,high_count
0,1214,1,10,23,29,33,37,40,16,1등,0,0,"10,23,29,33,37,40",172,4,2,1,5
1,1213,2,9,13,21,25,32,42,2,1등,1,2002006800,"9,13,21,25,32,42",142,4,2,3,3
2,1212,3,11,16,19,21,27,31,30,1등,1,2000000000,"11,16,19,21,27,31",125,5,1,4,2
3,1211,4,14,27,30,31,40,42,2,1등,0,0,"14,27,30,31,40,42",184,2,4,1,5
4,1210,5,16,24,29,40,41,42,3,1등,0,0,"16,24,29,40,41,42",192,2,4,1,5


### 4. Summary

This notebook keeps the Excel-based workflow as the stable default and retains the automated web-collection workflow as an in-progress feature. It then standardizes the raw columns, creates analysis-ready derived variables, and saves the cleaned result as a reusable CSV file.

At the end of this step, the project has a consistent processed dataset that can be used by the EDA, statistical testing, feature engineering, and modeling notebooks.